# Step 1. Convert landmark labels to Keypoint R-CNN format

Reads YOLO-style text labels (one line per vertebra: class, box centre, width, height and four corner points, all normalised) and writes one JSON file per image with absolute `boxes`, `keypoints` and `labels`, the format expected by `torchvision` Keypoint R-CNN.

**Input:** image folder and label folder. **Output:** JSON annotation folder.

Code comments are in Spanish.

In [ ]:
# Montar el Drive de Google
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import numpy as np
from google.colab.patches import cv2_imshow
import json

# Definir rutas
IMAGES = '/CARPETA_IMAGENES' # Colocar ruta de carpeta con imágenes de la base de datos
LABELS = '/CARPETA_LABELS' # Colocar ruta de carpeta con anotaciones de la base de datos
ANNOTATIONS = '/CARPETA_ANOTACIONES_NUEVAS' # Colocar ruta de carpeta donde se guardarán las anotaciones

file_image_example = os.path.join(IMAGES, 'IMAGEN_DE_PRUEBA')  # Coloca ruta de imagen a utilizar como ejemplo
img = cv2.imread(file_image_example)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(15,15))
plt.imshow(img)

In [ ]:
file_labels_example = os.path.join(LABELS, '/RUTA_DE_ANOTACION_PRUEBA')  # Coloca ruta de anotaciones de imagen a utilizar como ejemplo

with open(file_labels_example) as f:
    lines_txt = f.readlines()
    lines = []
    for line in lines_txt:
        parts = line.split()
        # Leer el cuadro delimitador
        class_id = int(parts[0])
        x_center = float(parts[1])
        y_center = float(parts[2])
        width = float(parts[3])
        height = float(parts[4])

        # Leer los puntos clave
        keypoints = []
        for i in range(5, len(parts), 3):
            x = float(parts[i])
            y = float(parts[i + 1])
            kp_id = int(parts[i + 2])
            keypoints.append([x, y, kp_id])

        lines.append((class_id, x_center, y_center, width, height, keypoints))

for idx, line in enumerate(lines):
    print("Anotación {}:".format(idx + 1), line)

Anotación 1: (0, 0.57265625, 0.221875, 0.15234375, 0.109375, [[0.5069582504970179, 0.18669938675388387, 2], [0.5077866302186879, 0.26369582992641044, 2], [0.6245858349900597, 0.26097027800490596, 2], [0.6245858349900597, 0.18261107931316434, 2]])
Anotación 2: (0, 0.56796875, 0.321875, 0.1703125, 0.10546875, [[0.5069582504970179, 0.2807304578904333, 2], [0.4995029821073559, 0.3611338103025347, 2], [0.620443986083499, 0.3645407604251839, 2], [0.6237574552683897, 0.27323520032706455, 2]])
Anotación 3: (0, 0.5609375, 0.41875, 0.1515625, 0.10078125, [[0.4995029821073559, 0.3829381439084219, 2], [0.4887342445328032, 0.4585718315617334, 2], [0.6154738071570577, 0.4565276778413736, 2], [0.6196156560636183, 0.3747615290269828, 2]])
Anotación 4: (0, 0.56015625, 0.51640625, 0.17578125, 0.11640625, [[0.49204771371769385, 0.47628781684382665, 2], [0.48542082504970174, 0.5580539656582174, 2], [0.621272365805169, 0.5478331970564186, 2], [0.6237574552683897, 0.47628781684382665, 2]])
Anotación 5: (0, 

In [ ]:
def classify_and_group_keypoints(file_labels, file_image):
    try:
        img = cv2.imread(file_image)
        if img is None:
            raise ValueError("Error: No es posible cargar el archivo de imagen.")

        img_w, img_h = img.shape[1], img.shape[0]

        with open(file_labels) as f:
            lines_txt = f.readlines()
            lines = []
            for line in lines_txt:
                parts = line.split()
                # Leer el cuadro delimitador
                class_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])

                # Convertir coordenadas normalizadas a absolutas
                x_min = round((x_center - width / 2) * img_w)
                y_min = round((y_center - height / 2) * img_h)
                x_max = round((x_center + width / 2) * img_w)
                y_max = round((y_center + height / 2) * img_h)
                bbox = [x_min, y_min, x_max, y_max]

                # Leer los puntos clave (4 puntos clave por anotación)
                keypoints = []
                for i in range(5, len(parts), 3):
                    x = round(float(parts[i]) * img_w)
                    y = round(float(parts[i + 1]) * img_h)
                    kp_id = int(parts[i + 2])
                    keypoints.append([x, y, kp_id])

                lines.append((bbox, keypoints))

        grouped_keypoints = []
        bboxes = []

        for bbox, keypoints in lines:
            bboxes.append(bbox)
            grouped_keypoints.append(keypoints)

        keypoint_rcnn_format = {
            'boxes': bboxes,
            'keypoints': grouped_keypoints,
            'labels': [0] * len(bboxes)  # Asumiendo que el 'label' para todos los cuadros es 0
        }

        return keypoint_rcnn_format

    except Exception as e:
        print(f"Error: {str(e)}")
        return None

# Ejemplo de uso
keypoint_rcnn_format = classify_and_group_keypoints(file_labels_example, file_image_example)
print(keypoint_rcnn_format)

{'boxes': [[318, 107, 415, 177], [309, 172, 418, 240], [310, 236, 408, 300], [302, 293, 415, 368], [308, 342, 455, 448], [316, 47, 432, 114]], 'keypoints': [[[324, 119, 2], [325, 169, 2], [400, 167, 2], [400, 117, 2]], [[324, 180, 2], [320, 231, 2], [397, 233, 2], [399, 175, 2]], [[320, 245, 2], [313, 293, 2], [394, 292, 2], [397, 240, 2]], [[315, 305, 2], [311, 357, 2], [398, 351, 2], [399, 305, 2]], [[312, 389, 2], [370, 417, 2], [421, 391, 2], [386, 360, 2]], [[323, 59, 2], [325, 105, 2], [403, 109, 2], [398, 54, 2]]], 'labels': [0, 0, 0, 0, 0, 0]}

In [ ]:
def visualize_results(image_path, bboxes, grouped_keypoints):
    img = cv2.imread(image_path)

    for bbox in bboxes:
        cv2.rectangle(img, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), 2)

    for group_keypoints in grouped_keypoints:
        for kp in group_keypoints:
            cv2.circle(img, (kp[0], kp[1]), 5, (0, 0, 255), -1)

    cv2_imshow(img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

bboxes = keypoint_rcnn_format['boxes']
grouped_keypoints = keypoint_rcnn_format['keypoints']
visualize_results(file_image_example, bboxes, grouped_keypoints)

In [ ]:
def process_images_and_save_annotations(images_path, labels_path, annotations_path):
    files_names = [file.split('.jpg')[0] for file in os.listdir(images_path) if file.endswith('.jpg')]

    for file in files_names:
        file_labels = os.path.join(labels_path, file + ".txt")
        file_image = os.path.join(images_path, file + ".jpg")

        keypoint_rcnn_format = classify_and_group_keypoints(file_labels, file_image)

        if keypoint_rcnn_format is not None:
            annotations = {
                'boxes': keypoint_rcnn_format['boxes'],
                'keypoints': keypoint_rcnn_format['keypoints'],
                'labels': keypoint_rcnn_format['labels']
            }

            os.makedirs(annotations_path, exist_ok=True)

            with open(os.path.join(annotations_path, file + '.json'), "w") as f:
                json.dump(annotations, f)
        else:
            print(f"Error processing file: {file}")

# Procesar las imágenes y guardar las anotaciones
process_images_and_save_annotations(IMAGES, LABELS, ANNOTATIONS)